# 第13章 暂态稳定分析

> Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 13
> Examples 13.1 - 13.4

## 本章概述

暂态稳定分析研究电力系统在遭受大扰动后能否保持同步运行。主要工具:

1. **等面积定则** (Equal Area Criterion): 图解方法，快速判断 SMIB 稳定性
2. **时域仿真** (Time-domain Simulation): 数值积分法，适用于多机系统
3. **临界清除时间** (CCT): 二分法搜索最大允许故障持续时间


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from psa4teaching.stability import (
    simulate_single_machine_infinite_bus_classic,
    simulate_single_machine_infinite_bus_detailed,
    simulate_multi_machine_classic,
)
from psa4teaching.data.kundur_two_area import create_kundur_two_area_system
from psa4teaching.network import build_ybus

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('Import OK')

---
## Example 13.1: 等面积定则

SMIB 系统: E'=1.2, V=1.0, X_pre=0.5, X_fault=1.5, X_post=0.6, H=5.0, Pm=0.8


In [ ]:
# ===== Ex 13.1: 等面积定则仿真 =====
E_prime = 1.2
V_inf = 1.0
X_pre = 0.5
X_fault = 1.5
X_post = 0.6
H = 5.0
D_smib = 0.0  # renamed to avoid conflict with multi-machine section
Pm = 0.8

delta_0 = np.arcsin(np.clip(Pm * X_pre / (E_prime * V_inf), -1, 1))
delta_cr = np.pi - delta_0
print(f'delta_0 = {np.degrees(delta_0):.2f} deg, delta_cr = {np.degrees(delta_cr):.2f} deg')

clearing_times = [0.10, 0.15, 0.20, 0.25]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, t_clear in zip(axes.flat, clearing_times):
    result = simulate_single_machine_infinite_bus_classic(
        E_prime=E_prime, V_infinity=V_inf, X_total=X_pre,
        H=H, D=D_smib, Pm=Pm, delta_0=delta_0,
        fault_time=0.0, fault_clearing_time=t_clear,
        X_total_fault=X_fault, X_total_post=X_post,
        t_end=5.0, dt=0.005)
    ax.plot(result.time, result.delta, 'b-', linewidth=1.5)
    ax.axhline(y=np.degrees(delta_cr), color='r', linestyle='--',
              label=f'delta_cr={np.degrees(delta_cr):.1f} deg')
    ax.axvline(x=t_clear, color='g', linestyle=':', label=f't_clear={t_clear:.2f}s')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('delta (deg)')
    ax.set_title(f't_clear = {t_clear:.2f}s -> {"STABLE" if result.stable else "UNSTABLE"}')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)

plt.suptitle('Equal Area Criterion: Effect of Fault Clearing Time', fontsize=14)
plt.tight_layout()
plt.savefig('examples/kundur/equal_area_demo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Example 13.3: 临界清除时间搜索


In [ ]:
# ===== Ex 13.3: CCT 二分搜索 =====
def find_cct(E_p, V_inf, X_p, X_f, X_po, H_v, D_v, Pm_v, d0, t_lo=0.01, t_hi=2.0, tol=0.001):
    for _ in range(30):
        t_mid = (t_lo + t_hi) / 2
        r = simulate_single_machine_infinite_bus_classic(
            E_p, V_inf, X_p, H_v, D_v, Pm_v, d0, 0.0, t_mid, X_f, X_po, 5.0, 0.005)
        if r.stable: t_lo = t_mid
        else: t_hi = t_mid
        if t_hi - t_lo < tol: break
    return (t_lo + t_hi) / 2

print('CCT Analysis:')
print(f'  X_fault=1.5, X_post=0.6: CCT = {find_cct(E_prime,V_inf,X_pre,X_fault,X_post,H,D_smib,Pm,delta_0):.4f} s')
print(f'  X_fault=1.0, X_post=0.5: CCT = {find_cct(E_prime,V_inf,X_pre,1.0,0.5,H,D_smib,Pm,delta_0):.4f} s')
print(f'  X_fault=2.0, X_post=0.8: CCT = {find_cct(E_prime,V_inf,X_pre,2.0,0.8,H,D_smib,Pm,delta_0):.4f} s')

---
## Example 13.1 (续): 等面积图 (P-delta 曲线)


In [ ]:
# ===== 等面积图 =====
delta_range = np.linspace(0, np.pi, 200)
Pe_pre = (E_prime * V_inf / X_pre) * np.sin(delta_range)
Pe_fault = (E_prime * V_inf / X_fault) * np.sin(delta_range)
Pe_post = (E_prime * V_inf / X_post) * np.sin(delta_range)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(np.degrees(delta_range), Pe_pre, 'b-', linewidth=2, label='Pre-fault (X=0.5)')
ax.plot(np.degrees(delta_range), Pe_fault, 'r-', linewidth=2, label='During fault (X=1.5)')
ax.plot(np.degrees(delta_range), Pe_post, 'g-', linewidth=2, label='Post-fault (X=0.6)')
ax.axhline(y=Pm, color='k', linestyle='--', linewidth=1, label=f'Pm={Pm}')
ax.axvline(x=np.degrees(delta_0), color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=np.degrees(delta_cr), color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('delta (deg)'); ax.set_ylabel('Power (pu)')
ax.set_title('Equal Area Criterion: P-delta Curves')
ax.set_xlim(0, 180); ax.set_ylim(0, 3.0)
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/equal_area_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Example 13.2: 两区域系统暂态稳定

节点 7 三相短路，0.1s 后切除，观察 Area 1 vs Area 2 功角摇摆。


In [ ]:
# ===== Ex 13.2: 两区域系统暂态仿真设置 =====
sys = create_kundur_two_area_system()
gens = sys['generators']

# 缩减 Ybus
full_ybus = build_ybus(sys['lines'], sys['transformers'])
gen_bus_nums = [g.bus for g in gens]
gen_idx = [full_ybus.bus_indices[b] for b in gen_bus_nums]
Y_pre = full_ybus.Ybus[np.ix_(gen_idx, gen_idx)]

print(f'Two-Area: {len(gens)} gens, Ybus: {Y_pre.shape[0]}x{Y_pre.shape[1]}')

E_primes = [1.05, 1.03, 1.03, 1.01]
H_vals = [g.H for g in gens]
D_vals = [0.0, 0.0, 0.0, 0.0]
Pm_vals = [7.0, 7.0, 7.19, 7.0]
delta_0m = np.radians([20, 15, 0, -5])  # multi-machine angles

print(f'H = {H_vals}')
print(f'delta0 = {[np.degrees(d) for d in delta_0m]} deg')

In [ ]:
# ===== 多机暂态仿真 =====
fault_time = 0.1
clear_time = 0.2

print('Running multi-machine simulation...')
result_multi = simulate_multi_machine_classic(
    E_primes=E_primes, H_list=H_vals, D_list=D_vals,
    Pm_list=Pm_vals, delta_0_list=list(delta_0m),
    Ybus_reduced=Y_pre,
    fault_time=fault_time, fault_clearing_time=clear_time,
    t_end=10.0, dt=0.01, stability_limit=180.0, verbose=True)

print(f'Max delta: {result_multi.max_delta:.2f} deg')
print(f'System: {"STABLE" if result_multi.stable else "UNSTABLE"}')

In [ ]:
# ===== 功角摇摆曲线 =====
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

colors = ['blue', 'cyan', 'red', 'orange']
for i in range(4):
    ax1.plot(result_multi.time, result_multi.delta[:, i],
            color=colors[i], linewidth=1.5,
            label=f'G{i+1} (Area {1 if i<2 else 2})')
ax1.axvline(x=fault_time, color='k', linestyle=':', alpha=0.5)
ax1.axvline(x=clear_time, color='k', linestyle='--', alpha=0.5)
ax1.set_ylabel('Rotor Angle (deg)')
ax1.set_title('Two-Area System: Absolute Rotor Angles')
ax1.grid(True, alpha=0.3); ax1.legend(loc='best')

# COI 角度差
coi1 = (result_multi.delta[:,0]*H_vals[0] + result_multi.delta[:,1]*H_vals[1]) / (H_vals[0]+H_vals[1])
coi2 = (result_multi.delta[:,2]*H_vals[2] + result_multi.delta[:,3]*H_vals[3]) / (H_vals[2]+H_vals[3])
ax2.plot(result_multi.time, coi1 - coi2, 'r-', linewidth=2)
ax2.axvline(x=fault_time, color='k', linestyle=':', alpha=0.5)
ax2.axvline(x=clear_time, color='k', linestyle='--', alpha=0.5)
ax2.axhline(y=0, color='gray', linewidth=0.5)
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Angle Diff (deg)')
ax2.set_title('Inter-Area Angle (COI1 - COI2)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('examples/kundur/two_area_transient.png', dpi=150, bbox_inches='tight')
plt.show()
print('Inter-area oscillation visible in the post-fault response')

---
## Example 13.4: 阻尼对暂态稳定的影响


In [ ]:
# ===== Ex 13.4: 阻尼系数对比 =====
D_vals_test = [0.0, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, Di in zip(axes.flat, D_vals_test):
    r = simulate_single_machine_infinite_bus_classic(
        E_prime, V_inf, X_pre, H, Di, Pm, delta_0,
        0.0, 0.15, X_fault, X_post, 5.0, 0.005)
    ax.plot(r.time, r.delta, 'b-', linewidth=1.5)
    ax.axhline(y=np.degrees(delta_cr), color='r', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('delta (deg)')
    ax.set_title(f'D = {Di:.1f} -> {"STABLE" if r.stable else "UNSTABLE"}')
    ax.grid(True, alpha=0.3)

r5 = simulate_single_machine_infinite_bus_classic(
    E_prime, V_inf, X_pre, H, 5.0, Pm, delta_0, 0.0, 0.15, X_fault, X_post, 5.0, 0.005)
print(f'D=5: first swing max={np.max(r5.delta):.1f} deg, final={r5.delta[-1]:.1f} deg')
plt.suptitle('Effect of Damping on Transient Stability', fontsize=14)
plt.tight_layout()
plt.savefig('examples/kundur/damping_effect.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 经典模型 vs 详细模型


In [ ]:
# ===== 经典 vs 详细模型对比 =====
r_cl = simulate_single_machine_infinite_bus_classic(
    E_prime, V_inf, X_pre, H, 0.0, Pm, delta_0,
    0.0, 0.15, X_fault, X_post, 5.0, 0.005)

r_det = simulate_single_machine_infinite_bus_detailed(
    E_prime_0=E_prime, V_infinity=V_inf, X_total=X_pre,
    Xd=1.8, Xd_prime=0.3, Xq=1.7, Td0_prime=8.0,
    H=H, D=0.0, Pm_0=Pm, delta_0=delta_0, Efd_0=2.0,
    Ka=50, Ta=0.05, Te=0.3,
    fault_time=0.0, fault_clearing_time=0.15,
    X_total_fault=X_fault, t_end=5.0, dt=0.005)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(r_cl.time, r_cl.delta, 'b-', linewidth=2, label='Classic (2nd)')
ax1.plot(r_det.time, r_det.delta, 'r--', linewidth=2, label='Detailed (3rd+Exc)')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('delta (deg)')
ax1.set_title('Rotor Angle Comparison'); ax1.grid(True, alpha=0.3); ax1.legend()

if r_det.Efd is not None:
    ax2.plot(r_det.time, r_det.Efd, 'g-', linewidth=2)
    ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Efd (pu)')
    ax2.set_title('Field Voltage (Detailed Model)')
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('examples/kundur/classic_vs_detailed.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Classic max: {np.max(r_cl.delta):.1f} deg, Detailed max: {np.max(r_det.delta):.1f} deg')

---
## 本章小结

1. **等面积定则**: 直观判断 SMIB 暂态稳定
2. **CCT**: 临界清除时间是保护整定的关键参数
3. **阻尼 D**: 改善后续摇摆但不改变第一摆稳定性
4. **励磁系统**: 快速励磁可提高第一摆稳定裕度
5. **多机系统**: 区域间振荡 ~0.5 Hz 是最危险的模式
